# Drum: satu uji 30 detik

Bass 90 detik sudah dibekukan sebagai baseline dengan catatan noise. Notebook ini membuat eksperimen drum terpisah dari piano asli, tanpa mengubah atau mengulang bass.

Gunakan **VM yang runtime dan bobot ACE-Step-nya masih tersedia**. Salin sel ke notebook aktif jika perlu. Helper sudah tersemat; tidak perlu ZIP tools atau mengunduh bobot lagi pada VM yang lengkap. Jika VM baru kosong, jalankan hanya sel source/setup/runtime/bobot notebook 12 sampai selesai, lalu lanjut di sini; jangan menjalankan generasi bass notebook 12.

Run all untuk satu probe: Base / LEGO / CUDA FP32 / seed 0 / CFG 7 / 50 langkah. Dengarkan kick/snare/hi-hat, timing dan feel. Durasi 30 detik adalah titik awal drum; pilihan bass 90 detik belum diterapkan ke drum. DRY_RUN=True memeriksa plan tanpa GPU, instalasi, unduhan atau inferensi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid
from google.colab import files
DRY_RUN = False
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU dan gunakan VM dengan runtime/bobot ACE-Step yang sudah siap"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "drum_probe.py": {
    "sha256": "0a79c5a14a983308e2d8c29699241c1e8c05b3ded35b2c7df31f2a885859aa8a",
    "gzip_base64": "H4sIAAAAAAAC/6VY3W/bOBJ/z1/B5oXSniznA73iHOgh26aH4vauRbd7B1wQELRE2VzLopaknKRB/vebIakv29l2b/1gW+RwOJ+/mdHp6enHWpDLs5kRuaoLUuh2SxqtloK0RtYrYteCiAdpLD40sq5FQa7f3sx+tqIhb395d010W1u5Fenp6emJ3DZKW8L1quHaiO75V6Pqk1IrYM3tupJLEjY+wWNHZNolXJwLY7qVr7IpZSVOumdRidxqmbOyrSoGPFeEG4JiItGnzzc/33whGXk6IfChsqALynPVguz5DBUzs905TehWFaJye8KAFrD4erbkRsBWIXYyF7CXtwWniWfUaJFLI1UN62WluL28AFJZGws8RW1h2XGHRSNEYeji9uwunF21suB1LpjJeQWM36RneLQUWrhVuB/oX8OiKpF3waxiedPSxRfdisAk5431138wquIWXNDp5R22kWDzPFfbhtcSRUoJ+AW001yCwxJSc9tqXpGVVmonyL20aziUbxJiaq4F4eD6vFIGOK/lbM1tQkpVVeq+C4FG8lqRRuhS6S0qRMDnsJmQZq25ixTkUTzWfCtzk5K3leDaS7eW1iTEclAWHEfApZVx1KYBF5ClsPdC1J6RgKPv0JhE1dUjCK7IToHp4FmTrahUAToPpjcpOOn55OSkECXEVr7hKxGp1jYtaGDaHKMpXjgrgmAic45iJXhpCcQpBBh1m84gIdzS/8rmPfwGPnM8mNB7mqB9wagYCVlP++ETe3fz/qfrLzfvYoxGrvO13Al/J37AYOAK8BqsguTk1svgcizFxICwAbaQL6KYyxpu7FZXohaao+fn4gFM7z3rN3v2B5/xMeOlPcLw7OyMXTMXtgyD9mwOmrXVd7DHYGy1mIffjrdnBYrspLhPt80lJpp8mKzcLSZ8ZUk6G3cGilNpGBo2ihfBkum9llbsEyb9iZPAKnh7uOL3fOp9EFTGKPgTDv5eJzsLpfd85y3j/u1ZBD/fq7aP+eBSEUFi28SoVuciCQnwQ2Is1zY7S8SDyFsrsveQSCLkw+GB7BbhOGriFIygqh04wSnWoEZHLojvHCOPvym3ludrR+Z94vFeA97UDjA6mF/zi9d/TZx6zBWGnjhUEibqndSqxnDvTh3ZGs4BGgjurI0RMDrCKrVaiWKg3PKc3Su9Aai9H9OB5wHVRoQOS7VgUBcMQHLdix+w2HERGvzxWyuBrmwuL0anoXqwvmx0R9//9PH6y+UF+8/Hz/+4+TxRe+Cer0XBflVLtw0luZSrDG06p/7BzKclzUdXlwSeZkgiB7NIkWJtMVHYB3NBoREPNorjV5kvnEMkQtEwgvybV6240VrpiL7zXYEwwpJCllC9zJV/1hDvWB9UVZABpNy1LTDhpBb34eR8B8ewjPnw8DGUbjeF1BHaGtA8c0XPtRtMbUK4+mgVaJ/siUJI2xZKZsBMiL2XarG/lS68elNIo7WyWOY/1IVoBHyBzEMTNLDzrdAVliGMBBBjwFAokdCGzFwb4urcby2vpH3ESuUqPAA6lCwr9PzHT/+E8soluIs++4LTh/8UksaI4XX2+lv9ODgImPKCW56F1AvlI5pk57yvKjTggPtOitaLn12exYPL3VVp2xSIJd7GGW0qjv0eTVxZyrprh2MhbbJRCnmcCDY/lKRbGcxIe3QKvwfs/5Ld0tlsyx9mkBfBR+jgyzP4ms26FmvmW6yEvnHLoT7NCqAvhEYj6N7WYZMGCAvZEwSYYnKwsWsaWfNo1+CyAeT2LXEb/mAHSGc51JQJXkRTDAh2uu07vLs4+aHjcL64G6QbwCwK+wlAYXYEFr1wCeBIvnH5FI9VhLjv1dQCbFAH3/dEQxZnI+CIDh130JTEE1yZGranXAkbdUkM2NPxg9jAvKlEHQ3EtxQcbsAor7LzqVMOIeoGTuXYGSNWI3pXAp9cYgIXOsgDT9nhHeCwK2h5IVKyI6oetEt0lDrYRGQDckeeTQJ/k3+BNBNLeGoiDcEt4toG14Q4u4TxA+3iBhDcBzZ+cwRySOBhLlDcUp/7zFdXeveqz9aDrW9Z0oG9l8T3o6E1JIDfspS5swgpORSYYmRWSHC+NUlXstwRuFtYnBzRwNRT0Lvk2KY/R+8mcYgOBczv55+DdPETF13A/HQwaR2fs450trSDRLq4BEYwqGyYfWywPlRipSb1hVfjyWyPjyfLw6z29xDZrj46bwGKw9SBlcTFqdvgbSEV1m1Mm8XRzptCA7WsBKtx9Krk1yCrq44g7lrWGyyD3QLUXQBjy3qr+fVjjDtSjBVzwADQf9XCQBU2nsdxzOvHyPvTBecGInLnmsVNssN2sfNeCmVuawAOFoeBdp1bqJg+RbtQCO0FnSRN6Fo8dLgJG66DhMXgH+8tuc3XzMivLoXO97dRLw3IqbYuh2mMaYiA6NT7Vlr8Unc6dRLDvOZu7IUfSR0KzFEIPTpBTcHzatxXRoFwYhOfzaHkx+N0V1quZM2rPuHRDoF8vyDP/Xk3i8R/AjM+IE6QfA0Rg/ZpsS/zZhoV+kH+UFPBoFB5mEuBgJrzHhsgblwTC0k+7pDCBDVIuoOUKFgJoSiwVHzPHf341XPuF77NF4Pc6DwpjE1WHEIdx71oMt0dmYXTN3ESDdccjMbJebxn5OF1WApVPrqlZbltBHa7s1oZW0icKWfIS6AP3Dp+yUmnA5LGsMhLmpR0pyoAsewJxX7+vSEfsWGWLzjwq+QS5KvACsh86dbO/3axmdwCtojvjrYcxxvLoegnXe4cqRhJaN1Yl0zh9yXBXcj4iM324n3wTpyg6Y9Tde6JX7phVFB8F5qNq/fB7suMGsE3zE2JvKpUjqZgy0crpgxfpKKj9uoPTBJXe2/I9l213w6Kh1w0lvwI9erG/cXCzw1xAbf4ho9Dg5A44qykT1hSI/cUp4zhWzXGnhfkyS090/9Lo/4Mzkd72o2mx5EyvSJAbsx05vUjOWJn9bj4I1KcnAAidzplGWVsCynGGPVGarLuhXh6rVeuj/iETzoqhMm1dPJkjBUqZ8xL3KS8KBgP1BHMM9jW0wSNmOFrGkiOkmPj6d7ZMAeVcHh4dZOGmfr2/O4llh7kx0xD2Sl8Fl8dHgFTvEz/0jU4d4ZT7v15L/vZsSvCeEIT7jqpDEqBgkoInZUIQcKzJnXmxHPQXFw1UG9s5Gpt0W4bE/WvxXjq5lGehtmYp6BBmIdh0U3E3fzJ024ChYEXXwlkF5Bn/wNlwfNYIxkAAA=="
  },
  "electric_full_song.py": {
    "sha256": "69b9a31dacb3dd8720466cdffcfa7e88c1df12011bfb335755ac3cf3e038494c",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoRnRpGlWD2xpLHsTKeqBgMCS2ojEEBxoUy7muk79A37JP0uu8DiQkpx03oyMbH3/e63teM4NypZxfIoC4pCHE//+z//64fptBBnsQzLXIXiZ2gfC5WEcRXBSFEVUgSFCEScJlJUWZwGkYzEaRoHC3Ev40zm3sHBp0IWoryXQn5WRYnzMpUkMC6vklKtYYkkEjJZKVjj5PpiLB5VeZ9WpVBlISIVrJIUpoXi9NeL74qD4+nkNZwpDDJPXMGMIq3yUIowTUr5uRwLPMk6jWQMQ+L4R5GkMAT2DO+r5AE3T3MBy5XhPXx4B47jHKh1lualCPJVFuSFNN9hmm3N799hDfN7HZT35ndaHCzzdC0yaIvVQujma2tIUS2yPA1lUdQt2/on3p9XKLcZgEk338DfsbwM1rLIgrA+UVWpyPx+DDZ1+xeVLVUsDw4+nHw8E3Px+tV0Oj24/nB2c/YRPr8eCPjjFOG9XAf+RuaFShNnJo7HwlER/HCkxvDRAjDsQDNBEHtg96KU2dHm+A12SmfMi0Vyo0KJI8IqCnBKlstQ6ZWdJZBC+foVtqukKPNqLZMSO8wGhZRRAQ230zu94gouFySh9AtAHK78V29K05cyl9QO58Apb7A5XeIWkV+mfphV0Poxr6ReCWij1Oe4KIAWS6C1JWAbLl5uY6BDQ894GAH7lkEugjBM11mQKDypJ05ElG/HIgnKKg9ikcVV+ADLlEheDm1j/XGQYkUYyyCPt0BHQK0VbxuUZRA+ANMU94ioJC2lyGH/AHkC6T4ATlrEyDzALOI+yNdposLCG9jkRBREFmJdFQpgRMePkWtUAjyYpPlaNx7pO+VyBRwH6y7TOE4fkfqRDzMVJOl3xcAWvD/cG+gSR+MJo20SrPFIgPqng4ODSC6BK4swBULy8zQtXaSLsQD8A6ZyGc0vAUijGePCcc4VXlMschWtgE/vZfiAzG2YHEGySNMHsYrTRRAXyJ9ASRUQ6UYCNIEakgTPYpgaVoCzOHx43BooHPmNjjHycgko30h3RP3rIH8AtMMQ1wERsVSrYgIAkmVB1OwhXyM5lmkaFxOA1dqXmyCegGjysi0RapirrIQ+WVaZr0UW9vEGCA2YoCIXRYC+Nf7JYXwOeIljl7rEBIhpDQdUhY/M6o4AKzm1If70QXlRtWygKVSBMBII1GZ1WlHfux468uRnoOAIxHLudiGh18WlBs9LZw4UgPMD3/Esz9PcXTofrq4+im0AGIhUpmJ1LxbVQ5A0mGTMzsRXXPHJGXVBgM3UBpiMgMdLifgAeOqTIRiAQCOgUwBEpMLSQ4n4ILeFq2mLcAXCGVhzZJ0ZZxIkYJ7LS4zNUhNAHakGx266AQwXv8hE5gFqoqP3SFSFFmq9P4c80UPSdB1UI+sg81Hg+YcTvfpo3BkGEtPX2sjnMf6hM+pAWrWpxgKNF0Tc2uCPwZTLINoC4G7pxvbVm8nta8AmhvScjUyiNJ+cnJ4d3aA8P/beTDxo3EwWKplkW2DGxLGIs70SyoG9SxExZKlKgFO6KmNCnOsVwVKWMinSvLA3umPSuE9BnRBdgJyUkcu3hUs2l6t5I5aJqyeMxN/EscV0fQJ2TiIQPnIBGM+CLtF64qJQAincf3dx+RG4OVkBaRdBHNzD/8vKwFjN+tLS/vO9QFnh/Q4QcIFGXFB5Y3OpUY1AYgfdCooP72IuDlQoice1hCWY+YwXF6WsJiBuAThh217EOtDrOjdafPE8EBDSwW3TwiPJM58LBxQzb+/YpHBgCQw9ucbZfoDrLy2wFzKu1iIAJGT4v9/eIzQ98c8A4gTFSAFjSLbW5iBYAWKRLmDfvSB3IlUrj2JyPAWmA7PTN8rdX1Zx7CPPeirbJouxiIO4EqCXwdIUKxIBhfKcFm74ohoFpLvvbeCD4eYhQdzOAHnAiYBl1+Cho0FQLtjdWo04QO+8+jJH686WwmREoFnnpRlQOM6mXiCsfOGMyNIGayGd2fLcpSZvJUtaEMQa4Gcs6tYkvAf9CehttRbBOntUESiP0Uj8AyhHNBzH4hX8N6gSfgPQGfSefc4AwqCY0LKQKRibD+++iOvT98c/9GV/cw5935G+PrA1GjogA79IH9yHlcuyEkwPsMNlidADQ2oOtl5UobBOk/kxfKwClcyn3l/HoDE/+3UX+CmNxQEnjFWoSnRLVkdaFoP3wTZ0AUdPE7AsFluwhkC7Pqaoen8EsbIKwq2IwfQBxwO23wKpAgmUtbVBhjoQASKZ8G3s9Vz+e6VySY00coMAI7ODrtFcgm/QPn3Na26QbF2wocFiRkvY3YDmgxVHKAeRD9tdLpwOdBta2qN6CDonwKhg7iqghA0bGRvUEnyktmAnXwjOJ34SU7PCFD7MycRP89ZJ8RtAbQ/F62DzcU8q2EQDPqDgM/GGf7MR+9Pc3uMnRCZpHFoaVft0fHynaavxpjyQF+6tDXnXWS7XmVwh7zlH6DWCl4py+WiD/5d4FPrkRuUwizLhjYZtAOeI/BUahwenpUvdUiMQG4MlWhrOJgV5J+df8fBPzq5Fg1wvgYzH00M8FHlMR+GMPapw7RfHP8RSj2XOGN2NWY3N0eVhqACRVmD7z41U0SNB8OD6hryCReHqkUc18EmD6oF7MLh03sowRe++poW1KoDawvsZkT/q/Eh8Nb1PQOhAIV95v6e2mOVGLQnAfEXLruZ/0M5ZNcj/ll/BTncOPkiC7GD4sLgPXr35YSwec6A0n5z2P8a3ACWLyTTv7uJB3f0CPmTSabHbc9wSdPmlUck1BizXgE0063CNwNl1/GZE6wb7dG4tNfpXbOhJdx9PUSx0JMezQqKFbBJe6LE+3qcxCpAY6SNvYGNiVGBBgJiPCg2jNFcrlQSxfx8U6CYxWRhGZzAmyxR6kEQ8DCQUriVZiL18JsS+iIFRC8kypiNXwMl/9MFHyZXc6VM4S/TTy7m5KE1MUXY42hm1ZBKwOuou5nTN6ptAgZ0To9tLGHPxKrd6Wefu1qlXvhsBnxP52MZcG231coS3ZvGfEH/Q4trEVhNaM+5viFVv+mYvYm/IKOthS6RLNhB+JCSjwUY4hiOCdoad4LAgxPEka5VU4AK8gAeao30v8GCWNtt7RA4jwmoUqZFId2CH4rka6SY/hzLPSkNlRB/e+iFSaKmhc1oQpsYc6fTTh/l5AIY1D8fwEBruPA1tQkOmHhh9bGCE6XqNlwDb8k9UbH0ldrcPjLP6GN/DOQa03V3rrDyItJ9RfmCf0UG+ScsRnPQeHW2vt+zrP/KlG/XHS9hEDyR0SCoO5RCP1qbM4XOar1Z8RKoYfCuROHeInr2kCfqX92Z129HCSK0vOYemwh6t1nSqj8IupOE1MEV57+/N3sdgH1gGArhnFNSfdyYemvE0ndHCYQ0dhGoImtsbch7waBg1lktjmGDcGWjtgcMf9XA8pYzsQBh+e+BBAgcG68J1X5Efww4NRoovry7PkM4QFRjizYGaEEaNonu8R6GziNPwQczm9Yk8jEFoiiICnpndyLzgnjx4dGlmK8qmYblzgvNvUwcA6xqYH4p/1OdBmCukCEbFVx21/60VtNeYhq+vjbSftfGGoUvkdWinv592mKILGZQUiAdd40TpY9JqCIHAIv01PP8xyPGsNMY5QRY5Iu/KvghlYVQCxhpKUY4tTx7kdkLLi7UsgygoA8+5e2K6qc03tyEue0ETt7XbRrunahMzMtO+DudCdoKyvnlziIKAf4I5Bovqn8bWEJ/Njnpgywphwn4aWhru0kzVk/aBYfSk+V2DkcimNoHqlVpGEZ4TObG3kebPHq71LXvjO2yKgYVY+hiGgDHMhA4JDp/JH1rpc2AHC/y2lDTZKH8Ppcsk8jUvNfvohqG72IzR8JOM7D0a89jcHV066ADttodMVYJGgSY0g5KW92Matf8jP8sQbBufwxAUMDJOUBPS+CAx14rWCJjvOv6iU6ZgfcB3vOW8qdD68ghkUoDpEkqFTc6vX7+a6AgIJm5yFVlZFDvStc+9Yru4ykA+y2A97GbpKB6YwRuVpwlm02pvq99F8+yh86FR9smsnInrvDv333362T89OX1HMh6+/+XsY/P9/vrXn09O/352+VZ/nV5dnl/88vbig2MZrGnh6c1ucWkM7Vmbc9tBe6BXZRiTdvUBrs7Pf724PJs7x7DPxw8nlzfnVx/en324sXu0gG/B0N0TxsUjB5hhaMUePSAAmZeo5uz44tB8rVJMqjnNw3vbJqIGD7MZGNmt7ebnwrunn96eiCpp7GzMXlGoV2OOpP3p9afJ++sbsQzieBGED07rLDpF4IHZEsVgv2BoExhMf9ojcS5crbDbYB+cwfR/oI1rTOWec9b5HS/jWkt6J6FEuOgu64rIgj6lktfA3j5LFqRkv8lpuIWMl2NxGOQrkDaHhw+P+Kuf0uF2DHe6JkFO4VXOkc8GwWmXXKBtIuuoRCEQ1E7bIceTeBEWCwCVMgZ1rr01LMvx2M7ttUnK3wl7o7fq4wQdy8lvJ2fzJle/jKvi3jKrO8HcosoosfgsuLqAagMbwONvAunTLTRoGVo6wmP2619vN0YBGm3stwMBmoxglCEonP8z/7YoyJS66B63u0CXcHz0on3/RQRSg89M2gmm1tHZrqWMKaCev3wEXWs4RdLYAt5/CvzDHbd1oQRKPK4Wsdp6szRWOgfbf4vumWEj/tLZ+x4uYEAHBzTyL+IUBAnaxFpduuANbFIF/okp2/nOLhZiG5RqhY7YTOC4vqcVDkoOz85BwL69lAS7Rdp2bDk6ppHV50rnk9O2e980d4fpmMFQlAB0tMyVVoX6nFkcJK7ZUiuFcV1Kczu960412gnD6r+nC2MoYWDdsJpf2zxzU8GDzEX8PDe0wFVAOyx/INFWDY4OfbTLeOaYqIFV5k5B4sdH8eP0THULggA46yLajmpadBAMCA3g0ymTco1Wne/Xqprs5hqQjeAi+dPcdjdABNeQgAxM1BKgWG+4p8ZEizgN7YMOEDX0gHrmDTDGNfXNG/QPIbVerQt9LKFCjq5ykLcUP8jnli/BPRofZb5tRIUmPcNtzONdNGAkIiuxLFCe0U9EIGpmVG3WWj3CBMO7rIq5swQbArhozDPmS+cr6QT6QkGJppfvP4EHRU2t4hLUpWwSojSKrbN/I2EZcxxtDJ+CPb2cZJhWGCmnJOSxnYp4UXLh/yN3VeehAhCh6zk6RuyTzb/S4Z/GnQTV/jCdte4fTkqpZTcdBaYQHWJflOuKqTMC/49KD9AkyjdSF5AGYVkrIF6dV3QM+kAOPAQr6bYcqJejCN2LNtLJ0XDrCkk/xoq6hLz9cVM46cM5N0o+euvsNaLQddbqM5UgmJH43Rpk6eX/A8KomXxvdrOF2Vgt4GAxwJRWWnDr8T+9euiuuRvnIKUAriib+RcVjBRu23fThQgasnWdBqikoopL74vKHIRgDVtUdD2At/HQgvZoz23t/ZZSRqgL+jvaiBrEnYU88rp1Ba73ryo7xxKZGlR8X+czBas4+IjZrXr8xbX/9uz8V2C8t1zhASav2si2zbakmsgY5NiGHd9DDWc0ORGUYDPriRxqbPY308b1L8MpbQTN7AiFa2vrukbSqKBJK8RhmTmTnnxt3cIeWTAchhaZTqf+CeHcxyLh6USThRmqldZE/61jYKaqhhRKjVm3G0f5u5SZCESxBo0hMNqJupe8WAaHCJYYT7+6ej/BpWD9phx9HRRUQI7VIWFQgWY38ZNeIHwHjbVqqtCXJwuwcJ+jJhz7pxCRiZwM4b5bOdgiYatgbyepManTaI0NjVWpg1rdDP/hriT/2Fj52jLWxDmwCGYrgswFobXACrOZroi1il+Br4cmmmzqCyNfp1fv3198vBnvrTD4htAXayYyWn2tAqwp4IWsVjJq7VDH4rAQOkJL0KYqHYu2kMUaFsnoMi3PQVdGpqBDpzxx2hLbwcri6cbM4nAErWQVHBoTt/VmgLnQKl42E61yZ0yBWfl2exBmWnwMSOh6NLa395kJrTcpJnASqeUSq735QQUYDAUdEpw3E+hvFJHWsREHEv/s2vBZt+baBOrMtqM+juzrUdG1STW23s7oOnr97IWDv2m+/bactBZ6lGcikxzzGSznYS/zEISSHJb7xroALu3M6JGMF4FUxR8u462nfh3jmPmYwVU4TzPULflgmGnqeFS9RMOzHlZ/Vw5O4TOgonF3da4HK0fxXviECCsEVVKlFQXGmqdElLSKZAZGNLJy84wIqcG8IBK/nZxRwUKpKBYK6oPcEU+8C/LoEVAADkpJxG/ebcAGBVkv4RafIoEmSiTml7k0q5SEaM/Zk2QYUM+MygFPzkoLDZdatQIaY9EpWLRLOnGHrguHwYmEJpJZMG+nPL71BhbnaFUwG4hvLp3rX08uZ/Tm66vZ+Pa7drrou7unwuCUQtO/XH8S9aui56OefKyD7s5Xl2fi+uTmZmbv3Ow5814vn3TB7viZw+noOVNo82KtprEdZ7SKRdA671eM6xJkUok+CRvftzQjORRHnHYii/8IZzlNXoEHAM5a9v+oiQwSqVBojAWfzJc+uWMY52zoplZkTQEH6MH5zkRP361oyO/W4QOb8AfFLgd2N5VPWnv2QmuWInKHQnY9W3bUUlI2kVojKfKvJSlH/o0JFDlYTYUvF5rhtw5o7wLLtGDk8YvrrlMSWejTUxW2FdZnQlLJJg057ticE3bCgGJv79vpnfWchp7JDMcwe3a5Y2EGDXSKoBqbpH6NA7/HVAHTApmeYUrKSB2Tkb8jdYIjYKXbThb6DkfUrNXtfA6gdkJEH4h3ZgcHbQa1VAxJ7VhYAOUwNpUggD4ihxeXgFPIsqSCiLtbhwehfhvq5alWpF0aFKNCbl4x9kLz+v0k5qLHQw8ndz2bHK7gsPPqNSw7KXfcsgT/h4L4qDdjuUrbrzuDuPUKs7cNGNTgZ/n8TFF9MVuSNYKrg6RD98pqAhcLGLr0G0hwz/D6ZjTeoRhYBR9+VMFKmq4nmyCx8J6RRST4ILdEfVQzT+YitIz1JxiMBlGewvdk7ujZRxMnrNtbDxJoP+DhvNCG64DdynbWxBT8tJiIyUfLHHpOS2fGh0X4bsrqXWCZtl+oL5qpjrsDEEY5COZ0TbwNo3TtGkHquct9SmrCteKtuNCEdhY10VvKi733nZJ42MsflsFk4aEaxB+ELfpBL+N4Nl2ShmHCBAQugg8+DXgoHwC3xudQoEEx74gpSos10SNurdaS8rqHhXxrmOajnkTrPsMYmkS8xnNaqVDHlFjTjborISmHWpDuXSEcuAjbPHgRAiPDEDYZvfxZUKOasOiEddJE5x5EO+NgANuuz24L9W4h013vAUu3QqqOU1mFWaO9muLlS7ZKdjuLtgqq7p4F2AUpGXyeteJXFejd2drcyh+ObBXRpCfqd621dkEnFQ09FNjN8Yciqc25W+VZdzu22r/YQG8rPrtrs+Y913M3xBhsb7NOjP35Kw27MI2NNq5F1byvsDGFR1Tsayqe6793RZstHaxzdbaV0+t1doatMxk88LNj4EW07cDIWmzBUWwtuHPUSxbWaZb967YH7V6WyJAZYd7jpT79gMGPmNw1oR3b37Un0xsWAFpPBGFmmYN0N21vxoa8mDbmz5PMt3qvnUTUwW6v8oVZ1GHa/V9kUHWAe5e933WAbImtp7ZiWG33jD/JO7MUvJk3qMa/FdIY7dgV+LekCgG5BjCWh4KMFX8RH4AEFL/0qDOLNW9OWPKzRSPNZO8PZqFfdB397hvOYiLyVNaC/pgpcfFO8hXZ2tfU40aSI5AYN/f9KA3Bv7dm4j8l4Ad6itv490gh82vKKcCWATD6fFeMQP+jC8Xt8d3elet/aqFee+9wjilY5zBhUcvn37UT1e7qyWTONLeY7p1pvbiyJ+O/UzR3rtaq7Dw3M1GydfBZras1vhYxFuyOHXSoimppCCmgkFPM48Kl9s+sozADE/URayK4+XSNHuFNqxRGr0t/4com54q+DdYk8gZ2TUar6pjGIHnYiRNqBFR1/yUK2fIJzA5F/YKKHxHpE0m2dgyJYL9BttNVzHCPOmVUn2hsL98cyuSOuLMds+TGpoDbZJOoWX+MOhE954Yk6qzmyFtjGaPv+x+C6xKw25wAWVZhDSDLWHIefC6L9NGXJk4++B+dGmjQVksAAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="drum_probe_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
single = importlib.util.module_from_spec(spec); spec.loader.exec_module(single)
sys.modules["electric_full_song"] = single
ROOT = single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = single.discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source bridge belum ditemukan. Gunakan VM aktif dari notebook 12 atau jalankan sel setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    single.model_python(ROOT)
spec = importlib.util.spec_from_file_location("drum_probe_embedded", TOOLS / "drum_probe.py")
drum = importlib.util.module_from_spec(spec); spec.loader.exec_module(drum)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload satu piano asli"
    SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
print("Runtime dipakai kembali:", ROOT)
print("Piano:", SOURCE_PATH)


## Satu generasi drum

START=0 memakai 30 detik awal piano. Folder output baru dibuat setiap uji; source/baseline bass tetap disimpan. Tidak memasukkan BPM atau birama perkiraan sebagai metadata terverifikasi.


In [ ]:
from IPython.display import Audio, display
START = 0
DRUM_OUT = ROOT / "runs" / ("drum_30_" + uuid.uuid4().hex[:10])
report = drum.generate(ROOT, SOURCE_PATH, DRUM_OUT, start=START, execute=not DRY_RUN)
print("Status:", report["status"], "|", DRUM_OUT)
if not DRY_RUN:
    for name in ("drums_preview.mp3", "mix_preview.mp3"):
        print(name)
        display(Audio(filename=str(DRUM_OUT / name), normalize=False))


## Simpan hasil dan kirim feedback

In [ ]:
for name in ("drum_result.zip", "drum_feedback.zip"):
    path = DRUM_OUT / name
    if path.is_file():
        files.download(str(path))
if DRY_RUN:
    print("Plan saja; tidak ada audio model atau ZIP hasil yang dibuat")
